# Conventional augmentation experiment
Run on Kaggle with Skin Cancer ISIC, ISIC 2019 Training Ground Truth, and ISIC 2019 Metadata attached and a GPU enabled. The single code cell rebuilds the verified split and checks its frozen membership fingerprint before training.

The CNN architecture, class weights, learning rate, batch size, epoch budget and early-stopping rule match the recovered baseline. Training augmentation: horizontal/vertical flips, rotation up to 0.05 turns (18 degrees), zoom factor 0.10, reflection filling. Validation uses original resized images; test images are not evaluated.

The comparison is an initial single-run experiment. The same seed does not guarantee identical random trajectories or fully deterministic GPU results; repeated seeds are needed for a robust conclusion.

Keras documentation: https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_rotation/ and https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_zoom/


Import the split dataset and inspect

In [1]:
#Import all necessary libraries
from pathlib import Path
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

In [2]:
MAPPING = {
    **dict.fromkeys(['NV', 'BKL', 'DF', 'VASC'], 'benign'),
    **dict.fromkeys(['MEL', 'BCC', 'SCC', 'AK'], 'malignant')
}

fingerprint_path = Path.cwd().parent / "artifacts" / "dataset_fingerprint.json"

with open(fingerprint_path) as f:
    fingerprint_data = json.load(f)
fingerprint = fingerprint_data["fingerprint"]

In [3]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_binary_split.csv"
out = Path.cwd().parent /'all_augmentations_results'
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

       image_id                                               path      label  \
0  ISIC_0000001   e:\Nová složka\input\Test\nevus\ISIC_0000001.jpg     benign   
1  ISIC_0000002  e:\Nová složka\input\Test\melanoma\ISIC_000000...  malignant   
2  ISIC_0000006   e:\Nová složka\input\Test\nevus\ISIC_0000006.jpg     benign   
3  ISIC_0000007   e:\Nová složka\input\Test\nevus\ISIC_0000007.jpg     benign   
4  ISIC_0000008   e:\Nová složka\input\Test\nevus\ISIC_0000008.jpg     benign   

  lesion_id               group  split original_label  target  
0       NaN  image:ISIC_0000001  train             NV       0  
1       NaN  image:ISIC_0000002  train            MEL       1  
2       NaN  image:ISIC_0000006  train             NV       0  
3       NaN  image:ISIC_0000007  train             NV       0  
4       NaN  image:ISIC_0000008  train             NV       0  
Index(['image_id', 'path', 'label', 'lesion_id', 'group', 'split',
       'original_label', 'target'],
      dtype='object')
spli

In [4]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

In [5]:

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

class_names = ["benign", "malignant"]

class_to_id = {
    "benign": 0,
    "malignant": 1
}


def make_dataset(frame, training=False, seed=42):

    paths = frame["path"].astype(str).values

    labels = frame["label"].map(
        class_to_id
    ).astype("float32").values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    def load_image(path, label):

        image = tf.io.read_file(path)

        image = tf.io.decode_image(
            image,
            channels=3,
            expand_animations=False
        )

        image.set_shape([None, None, 3])

        image = tf.image.resize(image, IMG_SIZE)

        image = tf.cast(image, tf.float32) / 255.0

        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=seed,
            reshuffle_each_iteration=True
        )

    return ds


In [6]:
y_train = train_df["label"].map(class_to_id).to_numpy()
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print(class_weights)
class_to_num = class_to_id
num_to_class = {v: k for k, v in class_to_num.items()}

{0: 0.9814126394052045, 1: 1.0193050193050193}


CNN

In [7]:
def build_cnn(filters, kernel_size, dense_units, dropout_rate):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(128, 128,3)),

        tf.keras.layers.Conv2D(filters[0], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[1], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[2], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[3], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(dense_units, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    return model

In [8]:
def build_augmentation(seed):

    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip(
            "horizontal_and_vertical", seed=seed
        ),

        tf.keras.layers.RandomRotation(
            0.05, fill_mode="reflect", seed=seed + 1
        ),

        tf.keras.layers.RandomZoom(
            0.10, fill_mode="reflect", seed=seed + 2
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.10,
            width_factor=0.10,
            fill_mode="reflect",
            seed=seed + 3
        ),

        tf.keras.layers.RandomCrop(
            height=110, width=110, seed=seed + 4
        ),

        tf.keras.layers.Resizing(128, 128),

        tf.keras.layers.RandomBrightness(
            0.15, value_range=(0, 1), seed=seed + 5
        ),

        tf.keras.layers.RandomContrast(
            0.15, seed=seed + 6
        )

    ], name=f"conventional_augmentation_seed_{seed}")


In [9]:

def make_mixed_dataset(
    df,
    augmentation,
    seed,
    augmented_fraction,
    training,
    batch_size
):

    n_original = len(df)

    # Calculate number of augmented images
    n_augmented = int(round(
        n_original * augmented_fraction /
        (1 - augmented_fraction)
    ))

    rng = np.random.default_rng(seed)

    augmented_indices = rng.choice(
        n_original,
        size=n_augmented,
        replace=True
    )

    augmented_df = df.iloc[augmented_indices].copy()

    # Original images (unbatched)
    original_ds = make_dataset(
        df,
        training=training,
        seed=seed
    )

    # Images selected for augmentation (unbatched)
    augmented_ds = make_dataset(
        augmented_df,
        training=False,
        seed=seed + 1000
    )

    # Apply augmentation
    augmented_ds = augmented_ds.map(
        lambda images, labels: (
            augmentation(images, training=True),
            labels
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Combine original and augmented images
    final_ds = original_ds.concatenate(augmented_ds)

    # Shuffle combined dataset during training
    if training:
        final_ds = final_ds.shuffle(
            buffer_size=n_original + n_augmented,
            seed=seed,
            reshuffle_each_iteration=True
        )

    # Batch exactly once
    final_ds = final_ds.batch(batch_size)

    # Prefetch
    final_ds = final_ds.prefetch(tf.data.AUTOTUNE)

    return final_ds


In [10]:

best_filters = (32, 64, 128, 256)
best_kernel_size = 5
best_dense_units = 128
best_dropout_rate = 0.45
best_learning_rate = 0.0001

# Run the model
results = []

for seed in [42, 43, 44]:

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    model = build_cnn(
        filters=best_filters,
        kernel_size=best_kernel_size,
        dense_units=best_dense_units,
        dropout_rate=best_dropout_rate
    )

    init_hash = sha256(
        b''.join(w.tobytes() for w in model.get_weights())
    ).hexdigest()

    # Preserve initial arrays for paired binary augmentation comparisons
    np.savez_compressed(
        out / f'initial_weights_seed{seed}.npz',
        *model.get_weights()
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=best_learning_rate
        ),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    run = f'binary_aug_seed{seed}'
    checkpoint = out / f'{run}.keras'

    # ==========================================
    # BUILD AUGMENTATION PIPELINE
    # ==========================================

    augmentation = build_augmentation(seed)

    # ==========================================
    # CREATE TRAINING AND VALIDATION DATASETS
    # ==========================================

    # Training: 50% original + 50% augmented
    train_ds = make_mixed_dataset(
        train_df,
        augmentation=augmentation,
        seed=seed,
        augmented_fraction=0.50,
        training=True,
        batch_size=BATCH_SIZE
    )

    # Validation during training:
    # 75% original + 25% augmented
    val_ds = make_mixed_dataset(
        val_df,
        augmentation=augmentation,
        seed=seed + 1,
        augmented_fraction=0.25,
        training=False,
        batch_size=BATCH_SIZE
    )

    # ==========================================
    # CREATE ORIGINAL EVALUATION DATASETS
    # ==========================================

   # Original, unaugmented training images
    train_eval_ds = make_dataset(
        train_df,
        training=False,
        seed=seed
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


    # Original, unaugmented validation images
    val_eval_ds = make_dataset(
        val_df,
        training=False,
        seed=seed
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


    # Original, unaugmented test images
    test_ds = make_dataset(
        test_df,
        training=False,
        seed=seed
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # ==========================================
    # TRAIN CNN
    # ==========================================

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        class_weight=class_weights,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                str(checkpoint),
                monitor='val_loss',
                save_best_only=True
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=2
    )

    best = tf.keras.models.load_model(checkpoint)

    # ==========================================
    # EVALUATION: TRAINING, VALIDATION AND TEST
    # ==========================================

    # Predict on all three original datasets
    train_probability = best.predict(
        train_eval_ds,
        verbose=0
    ).reshape(-1)

    val_probability = best.predict(
        val_eval_ds,
        verbose=0
    ).reshape(-1)

    test_probability = best.predict(
        test_ds,
        verbose=0
    ).reshape(-1)

    # True labels
    y_train = train_df.target.to_numpy(dtype=int)
    y_val = val_df.target.to_numpy(dtype=int)
    y_test = test_df.target.to_numpy(dtype=int)

    # Check that both classes are present
    for name, labels in [
        ('Training', y_train),
        ('Validation', y_val),
        ('Testing', y_test)
    ]:

        if len(np.unique(labels)) != 2:
            raise ValueError(
                f'{name} set must contain both benign and malignant classes.'
            )

    # Check prediction dimensions
    assert len(train_probability) == len(y_train)
    assert len(val_probability) == len(y_val)
    assert len(test_probability) == len(y_test)

    # Binary predictions (threshold = 0.5)
    train_pred = (train_probability >= .5).astype(int)
    val_pred = (val_probability >= .5).astype(int)
    test_pred = (test_probability >= .5).astype(int)

    # ==========================================
    # VALIDATION METRICS
    # ==========================================

    tn_val, fp_val, fn_val, tp_val = confusion_matrix(
        y_val, val_pred, labels=[0, 1]
    ).ravel()

    val_metrics = dict(
        accuracy=float(
            accuracy_score(y_val, val_pred)
        ),

        macro_f1=float(
            f1_score(
                y_val, val_pred,
                labels=[0, 1],
                average='macro',
                zero_division=0
            )
        ),

        balanced_accuracy=float(
            balanced_accuracy_score(y_val, val_pred)
        ),

        malignant_sensitivity=float(
            tp_val / (tp_val + fn_val)
        ) if (tp_val + fn_val) else 0.0,

        specificity=float(
            tn_val / (tn_val + fp_val)
        ) if (tn_val + fp_val) else 0.0,

        roc_auc=float(
            roc_auc_score(y_val, val_probability)
        ),

        average_precision=float(
            average_precision_score(y_val, val_probability)
        )
    )

    # ==========================================
    # TEST METRICS
    # ==========================================

    tn_test, fp_test, fn_test, tp_test = confusion_matrix(
        y_test, test_pred, labels=[0, 1]
    ).ravel()

    test_metrics = dict(
        accuracy=float(
            accuracy_score(y_test, test_pred)
        ),

        macro_f1=float(
            f1_score(
                y_test, test_pred,
                labels=[0, 1],
                average='macro',
                zero_division=0
            )
        ),

        balanced_accuracy=float(
            balanced_accuracy_score(y_test, test_pred)
        ),

        malignant_sensitivity=float(
            tp_test / (tp_test + fn_test)
        ) if (tp_test + fn_test) else 0.0,

        specificity=float(
            tn_test / (tn_test + fp_test)
        ) if (tn_test + fp_test) else 0.0,

        roc_auc=float(
            roc_auc_score(y_test, test_probability)
        ),

        average_precision=float(
            average_precision_score(y_test, test_probability)
        )
    )

    # ==========================================
    # TRAINING METRICS
    # ==========================================

    train_metrics = dict(
        accuracy=float(
            accuracy_score(y_train, train_pred)
        ),

        macro_f1=float(
            f1_score(
                y_train, train_pred,
                labels=[0, 1],
                average='macro',
                zero_division=0
            )
        ),

        roc_auc=float(
            roc_auc_score(y_train, train_probability)
        ),

        average_precision=float(
            average_precision_score(y_train, train_probability)
        )
    )

    # ==========================================
    # COMBINED RESULTS
    # ==========================================

    row = dict(
        seed=seed,
        condition='binary_aug',

        epochs_trained=len(history.history['loss']),

        selected_epoch=int(
            np.argmin(history.history['val_loss']) + 1
        ),

        validation_loss=float(
            min(history.history['val_loss'])
        ),

        initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,
        test_evaluated=True,

        **{
            f'train_{k}': v
            for k, v in train_metrics.items()
        },

        **{
            f'validation_{k}': v
            for k, v in val_metrics.items()
        },

        **{
            f'test_{k}': v
            for k, v in test_metrics.items()
        }
    )

    # ==========================================
    # CLASSIFICATION REPORTS
    # ==========================================

    val_report = classification_report(
        y_val, val_pred,
        labels=[0, 1],
        target_names=['benign', 'malignant'],
        output_dict=True,
        zero_division=0
    )

    test_report = classification_report(
        y_test, test_pred,
        labels=[0, 1],
        target_names=['benign', 'malignant'],
        output_dict=True,
        zero_division=0
    )

    (out / f'{run}_metrics.json').write_text(
        json.dumps(
            dict(
                **row,
                validation_classification_report=val_report,
                test_classification_report=test_report
            ),
            indent=2
        )
    )

    # ==========================================
    # SAVE TRAINING HISTORY
    # ==========================================

    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1, len(d) + 1)
    ).to_csv(
        out / f'{run}_history.csv',
        index=False
    )

    
    # ==========================================
    # VALIDATION LOSS CURVE
    # ==========================================

    plt.figure(figsize=(8, 6))

    epochs_range = np.arange(
        1, len(history.history['val_loss']) + 1
    )

    best_epoch = int(
        np.argmin(history.history['val_loss']) + 1
    )

    best_val_loss = float(
        min(history.history['val_loss'])
    )

    # Plot validation loss
    plt.plot(
        epochs_range,
        history.history['val_loss'],
        color='darkorange',
        linewidth=2,
        marker='o',
        markersize=4,
        label='Validation loss'
    )

    # Mark epoch with the lowest validation loss
    plt.axvline(
        x=best_epoch,
        color='red',
        linestyle='--',
        linewidth=1.5,
        label=f'Best epoch: {best_epoch}'
    )

    # Highlight minimum validation loss
    plt.scatter(
        best_epoch,
        best_val_loss,
        color='red',
        s=80,
        zorder=5
    )

    # Annotate minimum validation loss
    plt.annotate(
        f'Minimum: {best_val_loss:.4f}\n'
        f'Epoch: {best_epoch}',
        xy=(best_epoch, best_val_loss),
        xytext=(10, 15),
        textcoords='offset points',
        fontsize=10,
        color='red'
    )

    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Validation Loss', fontsize=12)

    plt.title(
        f'Validation Loss During Training - Seed {seed}',
        fontsize=13
    )

    plt.xticks(epochs_range)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    # Save validation loss curve
    plt.savefig(
        out / f'{run}_validation_loss.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ==========================================
    # SAVE PREDICTIONS
    # ==========================================

    for split_name, df, probability, pred in [
        ('training', train_df, train_probability, train_pred),
        ('validation', val_df, val_probability, val_pred),
        ('test', test_df, test_probability, test_pred)
    ]:

        predictions = df[
            ['image_id', 'original_label', 'label', 'group']
        ].copy()

        predictions['malignant_probability'] = probability
        predictions['prediction'] = pred

        predictions.to_csv(
            out / f'{run}_{split_name}_predictions.csv',
            index=False
        )

    # ==========================================
    # CONFUSION MATRICES
    # VALIDATION AND TEST
    # ==========================================

    for split_name, y_true, predictions in [
        ('validation', y_val, val_pred),
        ('test', y_test, test_pred)
    ]:

        cm = confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1]
        )

        pd.DataFrame(
            cm,
            index=['benign', 'malignant'],
            columns=['benign', 'malignant']
        ).to_csv(
            out / f'{run}_{split_name}_confusion_matrix.csv'
        )

        plt.figure(figsize=(7, 6))

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'],
            annot_kws={'size': 16}
        )

        plt.xlabel('Predicted label', fontsize=12)
        plt.ylabel('True label', fontsize=12)

        plt.title(
            f'{split_name.capitalize()} Confusion Matrix - {run}',
            fontsize=14
        )

        plt.tight_layout()

        plt.savefig(
            out / f'{run}_{split_name}_confusion_matrix.png',
            dpi=300,
            bbox_inches='tight'
        )

        plt.close()

    # ==========================================
    # COMBINED ROC CURVE
    # TRAINING, VALIDATION AND TEST
    # ==========================================

    plt.figure(figsize=(8, 7))

    roc_datasets = [
        ('Training', y_train, train_probability, 'royalblue'),
        ('Validation', y_val, val_probability, 'darkorange'),
        ('Testing', y_test, test_probability, 'green')
    ]

    roc_auc_values = {}

    for split_name, y_true, probability, color in roc_datasets:

        fpr, tpr, thresholds = roc_curve(
            y_true,
            probability
        )

        roc_auc_value = auc(fpr, tpr)

        roc_auc_values[split_name.lower()] = float(roc_auc_value)

        plt.plot(
            fpr,
            tpr,
            color=color,
            lw=2,
            label=f'{split_name} ROC (AUC = {roc_auc_value:.3f})'
        )

    # Random classifier reference
    plt.plot(
        [0, 1],
        [0, 1],
        color='navy',
        lw=2,
        linestyle='--',
        label='Random classifier'
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)

    plt.title(
        f'ROC Curves - Training, Validation and Testing - Seed {seed}',
        fontsize=13
    )

    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        out / f'{run}_ROC_curve.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    # ==========================================
    # SAVE ROC METRICS
    # ==========================================

    roc_results = pd.DataFrame([{
        'seed': seed,
        'training_auc': roc_auc_values['training'],
        'validation_auc': roc_auc_values['validation'],
        'test_auc': roc_auc_values['testing']
    }])

    roc_results.to_csv(
        out / f'{run}_ROC_metrics.csv',
        index=False
    )

    # ==========================================
    # SAVE COMBINED RESULTS
    # ==========================================

    results.append(row)

    pd.DataFrame(results).to_csv(
        out / 'binary_augmented_results.csv',
        index=False
    )

    # ==========================================
    # ARCHIVE EXPERIMENTAL EVIDENCE
    # ==========================================

    archive_path = out / f'{run}_artifacts.zip'

    files_to_archive = [
        out / f'{run}_metrics.json',
        out / f'{run}_history.csv',
        out / f'{run}_validation_loss.png',

        out / f'{run}_training_predictions.csv',
        out / f'{run}_validation_predictions.csv',
        out / f'{run}_test_predictions.csv',

        out / f'{run}_validation_confusion_matrix.csv',
        out / f'{run}_validation_confusion_matrix.png',

        out / f'{run}_test_confusion_matrix.csv',
        out / f'{run}_test_confusion_matrix.png',

        out / f'{run}_ROC_curve.png',
        out / f'{run}_ROC_metrics.csv',

        out / f'initial_weights_seed{seed}.npz',

        checkpoint,
        data_path,
        out / 'binary_augmented_results.csv'
    ]

    with zipfile.ZipFile(
        archive_path,
        'w',
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for p in files_to_archive:

            if p.exists():
                archive.write(p, p.name)

            else:
                print(f'Warning: missing file {p}')

    print(f'Archive saved: {archive_path}')

    print(
        f'Archive size: '
        f'{archive_path.stat().st_size / (1024**2):.2f} MB'
    )

    # ==========================================
    # PRINT RESULTS
    # ==========================================

    print(f'\nRESULTS FOR SEED {seed}')
    print('-' * 45)

    print('TRAINING:', train_metrics)
    print('VALIDATION:', val_metrics)
    print('TEST:', test_metrics)

    print(f'Model saved: {checkpoint}')

    print(
        'Initial weights saved: '
        f'initial_weights_seed{seed}.npz'
    )

    print(
        'BINARY_RUN_COMPLETE',
        json.dumps(row),
        flush=True
    )

    # ==========================================
    # CLEANUP
    # ==========================================

    del model, best
    del train_ds, val_ds
    del train_eval_ds, val_eval_ds, test_ds

    gc.collect()

print('\nALL AUGMENTED CNN EXPERIMENTS COMPLETED')



Epoch 1/25
66/66 - 55s - 831ms/step - accuracy: 0.5166 - auc: 0.5333 - loss: 0.6926 - val_accuracy: 0.4883 - val_auc: 0.6515 - val_loss: 0.6928
Epoch 2/25
66/66 - 21s - 315ms/step - accuracy: 0.4976 - auc: 0.4986 - loss: 0.6934 - val_accuracy: 0.4989 - val_auc: 0.6190 - val_loss: 0.6925
Epoch 3/25
66/66 - 90s - 1s/step - accuracy: 0.5161 - auc: 0.5259 - loss: 0.6932 - val_accuracy: 0.5541 - val_auc: 0.6105 - val_loss: 0.6922
Epoch 4/25
66/66 - 27s - 409ms/step - accuracy: 0.5327 - auc: 0.5428 - loss: 0.6910 - val_accuracy: 0.5902 - val_auc: 0.6838 - val_loss: 0.6753
Epoch 5/25
66/66 - 108s - 2s/step - accuracy: 0.5653 - auc: 0.5849 - loss: 0.6833 - val_accuracy: 0.5754 - val_auc: 0.7115 - val_loss: 0.6705
Epoch 6/25
66/66 - 16s - 248ms/step - accuracy: 0.5634 - auc: 0.5999 - loss: 0.6695 - val_accuracy: 0.6306 - val_auc: 0.6580 - val_loss: 0.6793
Epoch 7/25
66/66 - 40s - 601ms/step - accuracy: 0.5687 - auc: 0.6144 - loss: 0.6665 - val_accuracy: 0.5817 - val_auc: 0.7212 - val_loss: 0.6